## 1️⃣ Lasso vs Ridge vs Elastic Net - Comparaison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LassoCV, RidgeCV, ElasticNetCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import time

# Créer un dataset simple pour démonstration
np.random.seed(42)
n_samples, n_features = 200, 50
X = np.random.randn(n_samples, n_features)
coef_true = np.zeros(n_features)
coef_true[:10] = np.random.randn(10) * 5  # Seulement 10 features utiles
y = X @ coef_true + np.random.randn(n_samples) * 0.5

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("="*70)
print("1️⃣  LASSO vs RIDGE vs ELASTIC NET")
print("="*70)
print("\n📊 Comparaison des 3 méthodes de régularisation:\n")
print("+" + "-"*68 + "+")
print(f"{'Méthode':<20} | {'Pénalité':<30} | {'Sélection Features'}")
print("+" + "-"*68 + "+")
print(f"{'Lasso (L1)':<20} | {'alpha * sum(|coef|)':<30} | {'OUI ✓'}")
print(f"{'Ridge (L2)':<20} | {'alpha * sum(coef²)':<30} | {'NON ✗'}")
print(f"{'Elastic Net (L1+L2)':<20} | {'Combinaison L1 + L2':<30} | {'OUI ✓'}")
print("+" + "-"*68 + "+")

# Entraîner les 3 modèles
alpha = 0.1
lasso = Lasso(alpha=alpha)
ridge = Ridge(alpha=alpha)
elastic = ElasticNet(alpha=alpha, l1_ratio=0.5)

lasso.fit(X_train, y_train)
ridge.fit(X_train, y_train)
elastic.fit(X_train, y_train)

# Évaluer
models = {'Lasso': lasso, 'Ridge': ridge, 'Elastic Net': elastic}
print("\n📈 Performance (alpha=0.1):\n")
for name, model in models.items():
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    sparsity = np.sum(model.coef_ == 0) / len(model.coef_) * 100
    print(f"{name:15s} | R² = {r2:.4f} | RMSE = {rmse:.4f} | Sparsité = {sparsity:.1f}%")

print("\n💡 Quand utiliser quoi?")
print("  - Lasso:      Quand tu veux SÉLECTIONNER les features importantes (haute sparsité)")
print("  - Ridge:      Quand toutes les features sont utiles (multicollinéarité)")
print("  - ElasticNet: Cas intermédiaire, combine les avantages des 2")

## 2️⃣ Lasso Standard vs LassoCV

In [ ]:
print("\n" + "="*70)
print("2️⃣  LASSO vs LASSOCV")
print("="*70)

print("\n🎯 Approche 1: Lasso avec ALPHA MANUEL\n")
alphas_to_try = [0.001, 0.01, 0.1, 1.0, 10.0]
results_manual = []

for alpha in alphas_to_try:
    model = Lasso(alpha=alpha, max_iter=10000)
    model.fit(X_train, y_train)
    r2 = r2_score(X_test, model.predict(X_test))
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    non_zero = np.sum(model.coef_ != 0)
    results_manual.append({'alpha': alpha, 'R2': r2_score(y_test, y_pred), 'RMSE': rmse, 'Non-Zero': non_zero})
    print(f"  Alpha = {alpha:<6.3f} | R² = {r2_score(y_test, y_pred):.4f} | RMSE = {rmse:.4f} | Features = {non_zero}")

print("\n🔄 Approche 2: LassoCV avec ALPHA AUTOMATIQUE\n")
lasso_cv = LassoCV(alphas=np.logspace(-3, 2, 100), cv=5, max_iter=10000)
lasso_cv.fit(X_train, y_train)
y_pred_cv = lasso_cv.predict(X_test)
r2_cv = r2_score(y_test, y_pred_cv)
rmse_cv = np.sqrt(mean_squared_error(y_test, y_pred_cv))
non_zero_cv = np.sum(lasso_cv.coef_ != 0)

print(f"  Alpha trouvé automatiquement: {lasso_cv.alpha_:.6f}")
print(f"  R² = {r2_cv:.4f} | RMSE = {rmse_cv:.4f} | Features = {non_zero_cv}")

print(f"\n💡 Avantages de LassoCV:")
print(f"  ✓ Pas besoin de tester manuellement")
print(f"  ✓ Validation croisée intégrée")
print(f"  ✓ Meilleur alpha généralement trouvé")
print(f"  ✓ Temps d'exécution raisonnable")

## 3️⃣ LassoLars - Algorithme LARS

In [ ]:
from sklearn.linear_model import LassoLars, LassoLarsCV

print("\n" + "="*70)
print("3️⃣  LASSCO vs LASSOLARS - Algorithmes différents")
print("="*70)

print("\n⚡ LassoLars = Least Angle Regression")
print("   Calcule EFFICACEMENT la régularisation path (tous les alphas)\n")

# Comparer vitesse
start = time.time()
lasso_cv = LassoCV(alphas=np.logspace(-3, 2, 100), cv=5, max_iter=10000)
lasso_cv.fit(X_train, y_train)
time_lasso_cv = time.time() - start
r2_lasso_cv = r2_score(y_test, lasso_cv.predict(X_test))

start = time.time()
lasso_lars_cv = LassoLarsCV(cv=5, max_iter=1000)
lasso_lars_cv.fit(X_train, y_train)
time_lars_cv = time.time() - start
r2_lars_cv = r2_score(y_test, lasso_lars_cv.predict(X_test))

print(f"Résultats de comparaison:\n")
print(f"  {'Méthode':<20} | {'Temps (ms)':<15} | {'R²':<10} | {'Alpha'}")
print(f"-" * 60)
print(f"  {'LassoCV':<20} | {time_lasso_cv*1000:<15.2f} | {r2_lasso_cv:<10.4f} | {lasso_cv.alpha_:.6f}")
print(f"  {'LassoLarsCV':<20} | {time_lars_cv*1000:<15.2f} | {r2_lars_cv:<10.4f} | {lasso_lars_cv.alpha_:.6f}")

speedup = time_lasso_cv / time_lars_cv
print(f"\n  LassoLars est {speedup:.1f}x plus RAPIDE! ⚡\n")

print(f"💡 Quand utiliser LassoLars?")
print(f"  ✓ Datasets moyens à grands (n_features > 1000)")
print(f"  ✓ Besoin de vitesse")
print(f"  ✓ Pas besoin d'une précision maximale")

## 4️⃣ SGDRegressor avec L1 - Stochastic Gradient Descent

In [ ]:
from sklearn.linear_model import SGDRegressor

print("\n" + "="*70)
print("4️⃣  SGDREGRESSOR avec L1 - Pour GRANDS DATASETS")
print("="*70)

print("\n🚀 SGD = Stochastic Gradient Descent")
print("   Optimal pour datasets TRÈS GRANDS (millions de samples)\n")

# SGD avec L1
sgd_l1 = SGDRegressor(penalty='l1', alpha=0.01, max_iter=1000, random_state=42)
sgd_l1.fit(X_train, y_train)
r2_sgd_l1 = r2_score(y_test, sgd_l1.predict(X_test))
rmse_sgd_l1 = np.sqrt(mean_squared_error(y_test, sgd_l1.predict(X_test)))
non_zero_sgd = np.sum(sgd_l1.coef_ != 0)

print(f"Résultats SGDRegressor (L1):\n")
print(f"  R² = {r2_sgd_l1:.4f}")
print(f"  RMSE = {rmse_sgd_l1:.4f}")
print(f"  Features non-zéro = {non_zero_sgd}")

print(f"\n💡 Caractéristiques de SGD:")
print(f"  ✓ Très rapide sur GRANDS datasets")
print(f"  ✓ Apprentissage online (mini-batches)")
print(f"  ✓ Bon pour streaming data")
print(f"  ✗ Moins précis sur petits datasets")
print(f"  ✗ Nécessite normalisation des données")

## 5️⃣ Lasso avec warm_start - Optimisation Progressive

In [ ]:
print("\n" + "="*70)
print("5️⃣  LASSO avec WARM_START - Optimisation Progressive")
print("="*70)

print("\n🔥 warm_start = Commencer l'optimisation à partir du dernier modèle\n")

# Sans warm_start
start = time.time()
alphas_test = np.logspace(-3, 2, 10)
models_cold = []
for alpha in alphas_test:
    model = Lasso(alpha=alpha, max_iter=1000)
    model.fit(X_train, y_train)
    models_cold.append(model)
time_cold = time.time() - start

# Avec warm_start
start = time.time()
model_warm = Lasso(max_iter=1000, warm_start=True)
models_warm = []
for alpha in alphas_test:
    model_warm.set_params(alpha=alpha)
    model_warm.fit(X_train, y_train)
    models_warm.append(model_warm)
time_warm = time.time() - start

print(f"Temps d'exécution:\n")
print(f"  Sans warm_start:  {time_cold*1000:.2f} ms")
print(f"  Avec warm_start:  {time_warm*1000:.2f} ms")
print(f"  Speedup:          {time_cold/time_warm:.2f}x ⚡")

print(f"\n💡 Utilité du warm_start:")
print(f"  ✓ Grille de search sur alphas")
print(f"  ✓ Fine-tuning d'hyperparamètres")
print(f"  ✓ Économise temps de calcul")

## 6️⃣ Comparaison Pratique - Benchmark Complet

In [ ]:
print("\n" + "="*70)
print("6️⃣  BENCHMARK COMPLET - Comparaison Finale")
print("="*70)

methods = {}

# 1. Lasso Standard
start = time.time()
lasso = Lasso(alpha=0.01, max_iter=10000)
lasso.fit(X_train, y_train)
methods['Lasso (alpha=0.01)'] = {
    'time': time.time() - start,
    'r2': r2_score(y_test, lasso.predict(X_test)),
    'rmse': np.sqrt(mean_squared_error(y_test, lasso.predict(X_test))),
    'sparsity': np.sum(lasso.coef_ == 0) / len(lasso.coef_) * 100
}

# 2. LassoCV
start = time.time()
lasso_cv = LassoCV(alphas=np.logspace(-3, 2, 50), cv=5, max_iter=10000, n_jobs=-1)
lasso_cv.fit(X_train, y_train)
methods['LassoCV'] = {
    'time': time.time() - start,
    'r2': r2_score(y_test, lasso_cv.predict(X_test)),
    'rmse': np.sqrt(mean_squared_error(y_test, lasso_cv.predict(X_test))),
    'sparsity': np.sum(lasso_cv.coef_ == 0) / len(lasso_cv.coef_) * 100
}

# 3. LassoLarsCV
start = time.time()
lasso_lars_cv = LassoLarsCV(cv=5, max_iter=1000)
lasso_lars_cv.fit(X_train, y_train)
methods['LassoLarsCV'] = {
    'time': time.time() - start,
    'r2': r2_score(y_test, lasso_lars_cv.predict(X_test)),
    'rmse': np.sqrt(mean_squared_error(y_test, lasso_lars_cv.predict(X_test))),
    'sparsity': np.sum(lasso_lars_cv.coef_ == 0) / len(lasso_lars_cv.coef_) * 100
}

# 4. SGDRegressor
start = time.time()
sgd = SGDRegressor(penalty='l1', alpha=0.01, max_iter=1000, random_state=42)
sgd.fit(X_train, y_train)
methods['SGDRegressor (L1)'] = {
    'time': time.time() - start,
    'r2': r2_score(y_test, sgd.predict(X_test)),
    'rmse': np.sqrt(mean_squared_error(y_test, sgd.predict(X_test))),
    'sparsity': np.sum(sgd.coef_ == 0) / len(sgd.coef_) * 100
}

# Tableau comparatif
print("\n📊 RÉSULTATS COMPARATIFS:\n")
print(f"{'Méthode':<20} | {'Temps (ms)':<12} | {'R²':<10} | {'RMSE':<10} | {'Sparsité %'}")
print("-" * 75)
for method, results in methods.items():
    print(f"{method:<20} | {results['time']*1000:<12.2f} | {results['r2']:<10.4f} | {results['rmse']:<10.4f} | {results['sparsity']:<10.1f}")

print("\n💡 RECOMMANDATIONS:\n")
print("  1️⃣  Pour PETITS datasets (< 10k samples):")
print("      → Utiliser LassoCV ou LassoLarsCV")
print("\n  2️⃣  Pour MOYENS datasets (10k - 1M samples):")
print("      → Utiliser LassoLarsCV (plus rapide)")
print("\n  3️⃣  Pour GRANDS datasets (> 1M samples):")
print("      → Utiliser SGDRegressor ou LassoCV avec n_jobs=-1")
print("\n  4️⃣  Pour fine-tuning d'alphas:")
print("      → Utiliser Lasso avec warm_start=True")

## 📋 RÉSUMÉ - Tableau Comparatif Final

In [ ]:
import pandas as pd

print("\n" + "="*90)
print("📋 TABLEAU RÉCAPITULATIF - Types de Lasso Regression")
print("="*90)

summary = pd.DataFrame([
    {
        'Type': 'Lasso Standard',
        'Sélection Alpha': 'Manuel ❌',
        'Vitesse': '⚡⚡⚡',
        'Précision': '⭐⭐⭐',
        'Meilleur Pour': 'Baseline simple',
        'Complexité': 'Bas'
    },
    {
        'Type': 'LassoCV',
        'Sélection Alpha': 'Auto (CV) ✓',
        'Vitesse': '⚡⚡',
        'Précision': '⭐⭐⭐⭐⭐',
        'Meilleur Pour': 'Petits à moyens datasets',
        'Complexité': 'Moyen'
    },
    {
        'Type': 'LassoLars',
        'Sélection Alpha': 'Manuel ❌',
        'Vitesse': '⚡⚡⚡⚡',
        'Précision': '⭐⭐⭐⭐',
        'Meilleur Pour': 'Regularization path complet',
        'Complexité': 'Moyen'
    },
    {
        'Type': 'LassoLarsCV',
        'Sélection Alpha': 'Auto (CV) ✓',
        'Vitesse': '⚡⚡⚡',
        'Précision': '⭐⭐⭐⭐',
        'Meilleur Pour': 'Vitesse + Précision',
        'Complexité': 'Moyen'
    },
    {
        'Type': 'SGDRegressor (L1)',
        'Sélection Alpha': 'Manual + tuning',
        'Vitesse': '⚡⚡⚡⚡⚡',
        'Précision': '⭐⭐⭐',
        'Meilleur Pour': 'Très grands datasets',
        'Complexité': 'Élevé'
    },
    {
        'Type': 'Lasso + warm_start',
        'Sélection Alpha': 'Manuel ❌',
        'Vitesse': '⚡⚡⚡⚡',
        'Précision': '⭐⭐⭐⭐',
        'Meilleur Pour': 'Grille d\'alphas',
        'Complexité': 'Moyen'
    },
    {
        'Type': 'ElasticNet / Ridge',
        'Sélection Alpha': 'Alternatif (pas L1)',
        'Vitesse': '⚡⚡⚡',
        'Précision': '⭐⭐⭐⭐',
        'Meilleur Pour': 'Autres régularisations',
        'Complexité': 'Bas'
    }
])

print("\n")
print(summary.to_string(index=False))

print("\n" + "="*90)
print("\n🎯 RECOMMANDATION FINALE:\n")
print("   → Commencez par LassoCV (meilleur rapport qualité/vitesse)")
print("   → Si trop lent → Utilisez LassoLarsCV")
print("   → Si TRÈS GRAND dataset → Utilisez SGDRegressor")
print("   → Si fine-tuning d'alphas → Utilisez Lasso + warm_start")
print("\n" + "="*90)